# 03 First engagement on leaving Melted

Counted on the 11-state trajectories, no Markov model. A transition is committed only once the new state
has been held for a minimum dwell (1, 5 or 10 frames = 0.1, 0.5, 1.0 ns); shorter visits are treated as
excursions from the previous state. Every exit from Melted is classified by the strand that engages:

* $\beta_2$-engaged: $\beta_2$ Anchored, $\beta_1$ Frustrated, $\beta_2$ Exploring
* $\beta_1$-engaged: $\beta_1$ Anchored, $\beta_2$ Frustrated, $\beta_1$ Exploring

Intervals are 95% bootstraps over whole trajectories (2000 resamples).

In [1]:
import numpy as np
import pandas as pd
import abeta_tips as T

FOLDS = ["nu_8EZD", "S_7Q4M"]
macro = {f: T.split_per_traj(T.macrostates(f), f) for f in FOLDS}

In [2]:
def dwell_filter(t, tau):
    """Switch to a new state only after it has been held for tau consecutive frames."""
    if tau <= 1:
        return t
    out = np.empty_like(t)
    cur, run = t[0], 1
    out[0] = t[0]
    for i in range(1, len(t)):
        run = run + 1 if t[i] == t[i - 1] else 1
        if run >= tau:
            cur = t[i]
        out[i] = cur
    return out


def melted_exits(traj, tau):
    """Destinations of all committed exits from Melted in one trajectory."""
    tc = dwell_filter(traj, tau)
    i = np.flatnonzero(np.diff(tc) != 0) + 1
    return tc[i][tc[i - 1] == T.MELTED]

In [3]:
rows = []
dest = {}
for f in FOLDS:
    for tau in (1, 5, 10):
        d = np.concatenate([melted_exits(t, tau) for t in macro[f]])
        dest[f, tau] = d
        b2 = np.isin(d, list(T.B2_ENGAGED)).mean()
        b1 = np.isin(d, list(T.B1_ENGAGED)).mean()
        rows.append({"fold": f, "dwell (ns)": tau * T.DT_NS, "exits": len(d),
                     "b2-engaged %": 100 * b2, "b1-engaged %": 100 * b1, "other %": 100 * (1 - b1 - b2)})
pd.DataFrame(rows).round(1)

,fold,dwell (ns),exits,b2-engaged %,b1-engaged %,other %
0,nu_8EZD,0.1,21394,86.3,11.5,2.2
1,nu_8EZD,0.5,559,91.2,8.6,0.2
2,nu_8EZD,1.0,249,92.4,7.2,0.4
3,S_7Q4M,0.1,13501,86.1,11.1,2.8
4,S_7Q4M,0.5,417,89.2,10.3,0.5
5,S_7Q4M,1.0,178,91.6,6.7,1.7


## Landing states at the 0.5 ns filter

In [4]:
land = {f: pd.Series(dest[f, 5]).map(dict(enumerate(T.MACRO_NAMES))).value_counts(normalize=True) * 100
        for f in FOLDS}
pd.DataFrame(land).fillna(0).round(1).sort_values("nu_8EZD", ascending=False)

,nu_8EZD,S_7Q4M
b2 Exploring,69.2,71.7
b2 Anchored,22.0,17.5
b1 Exploring,5.5,9.6
b1 Anchored,2.3,0.7
b2 Frustrated,0.7,0.0
Pure Native,0.2,0.0
Misregistered Trap,0.0,0.5


## Normalized to strand-engaging exits, with trajectory bootstrap (Fig. 2a)

In [5]:
def engagement_bootstrap(trajs, tau=5, n_boot=2000, seed=0):
    n1 = np.zeros(len(trajs)); n2 = np.zeros(len(trajs))
    for k, t in enumerate(trajs):
        d = melted_exits(t, tau)
        n1[k] = np.isin(d, list(T.B1_ENGAGED)).sum()
        n2[k] = np.isin(d, list(T.B2_ENGAGED)).sum()
    tot = n1 + n2
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, len(trajs), len(trajs))
        boot[b] = 100 * n2[idx].sum() / tot[idx].sum()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return dict(strand_exits=int(tot.sum()), b2=100 * n2.sum() / tot.sum(), b2_lo=lo, b2_hi=hi,
                b1=100 * n1.sum() / tot.sum(), b1_lo=100 - hi, b1_hi=100 - lo)

pd.DataFrame({f: engagement_bootstrap(macro[f]) for f in FOLDS}).round(1)

,nu_8EZD,S_7Q4M
strand_exits,558.0,415.0
b2,91.4,89.6
b2_lo,87.6,85.4
b2_hi,94.4,93.5
b1,8.6,10.4
b1_lo,5.6,6.5
b1_hi,12.4,14.6
